# 04 · Auditoria visual dos experimentos

**Objetivo:** verificar a consistência das evidências já produzidas, reconhecer limitações da coleta e compreender *por que* o Router altera a atribuição dos pools Tornado Cash.

**Duração da leitura:** 4 etapas: **fontes → checks → gráficos → conclusão**. As células de código foram reduzidas; o foco é interpretar resultados.

**Importante:** este notebook **audita relatórios históricos já produzidos** (12.02B e reprodução consolidada 01). Não recalcula a blockchain nem prova independência das fontes. Os dados são obtidos **do seu próprio Drive** pelos IDs, sem GitHub Token ou atalho do Bruno.

**Perguntas que você deve saber responder ao professor:**
1. Por que há operações via Router em 2026 e diretas em 2020?
2. O que significa “paridade das features” e o que **não** significa?
3. Como distinguir uma janela de 1 dia de um CSV de 7 dias?
4. Como evitar confundir reprodução interna com generalização temporal?

## 1 · Fontes e preparação

**O que vamos ler:** (A) reconciliação por pool/dia, (B) paridade das quatro features, (C) evidência de cobertura, (D) validação cruzada nova e antiga e (E) resumo dos limiares em 2020.

**Ação:** execute a célula abaixo, autorize a conta Google que tem acesso à pasta `Projeto_Mixers_Ethereum` e observe a tabela de arquivos. Não escrevemos sobre os datasets.

In [ ]:
#@title 1. Conectar ao Drive e carregar evidências (somente leitura)
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from IPython.display import display
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import json, io, hashlib
from datetime import datetime, timezone

auth.authenticate_user()
credenciais, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
api = build('drive', 'v3', credentials=credenciais, cache_discovery=False)
ARQUIVOS = {
    'atribuicao': '1f98EfSSaGG-S1odChy0BkYSJyDDvDJ-p',
    'paridade': '1Vuo98PnMKRoL1t4j9MHmkQz4s8bGfbfS',
    'fontes': '1oOO0lrLlkQkDm53wLbXxf8Ws8QJ8-Ytw',
    'colisoes': '1ENfHRsotOn8AVhRCk1D4h-6KNtNjHYqu',
    'cv_novo': '1_fXFj9Glynv3OFFCSfXPDxwA6UAFXg2Q',
    'cv_legado': '1d9s595kCgSakPrqJk6MgCcka_YRLbr-x',
    'limiares': '1giUxsAxWG-YFuTX6UuXL4QvN4NXu6aMB',
}
def baixar(id_drive):
    memoria = io.BytesIO()
    requisicao = api.files().get_media(fileId=id_drive, supportsAllDrives=True)
    download = MediaIoBaseDownload(memoria, requisicao)
    concluido = False
    while not concluido:
        _, concluido = download.next_chunk()
    return memoria.getvalue()

brutos = {nome: baixar(fid) for nome, fid in ARQUIVOS.items()}
tabelas = {nome: pd.read_csv(io.BytesIO(conteudo)) for nome, conteudo in brutos.items()}
display(pd.DataFrame([{'fonte': nome, 'linhas': len(tabelas[nome]), 'SHA-256': hashlib.sha256(conteudo).hexdigest()[:12]+'…'}
                      for nome, conteudo in brutos.items()]))

## 2 · Checks que realmente podemos fazer

Uma **linha de paridade** representa uma combinação *dia + pool* em que recalculamos as features e comparamos aos resultados salvos. A auditoria anterior encontrou **7 combinações com referência**: quatro de 2020 e três de 2026.

**Limitações essenciais:** em 2020 não temos arquivos de chamadas internas equivalentes; logo, *não observamos chamadas* não significa *provar ausência de chamadas*. Em 2025, a análise de 15/03 possui apenas 3 transações coincidentes naquele dia e não tem janela processada de referência; não é falha de paridade. Seu CSV de chamadas abrange 15–21/03, embora o nome contenha apenas 15/03.

A célula abaixo testa os próprios relatórios e **interrompe** se seus números fundamentais divergirem.

In [ ]:
#@title 2. Verificar consistência e resumir o que foi auditado
par = tabelas['paridade'].copy()
atr = tabelas['atribuicao'].copy()
cv = tabelas['cv_novo'].copy()
leg = tabelas['cv_legado'].copy()
lim = tabelas['limiares'].copy()

confirmadas = par[par['status'].eq('PARIDADE_CONFIRMADA')]
sem_ref = par[par['status'].eq('SEM_FEATURE_REFERENCIA')]
assert len(confirmadas) == 7, 'Esperávamos 7 combinações com referência.'
assert confirmadas['todas_features_batem'].astype(str).str.lower().eq('true').all()
assert len(sem_ref) == 1 and sem_ref.iloc[0]['periodo'] == '2025-03-15'
assert tabelas['colisoes'].empty, 'Há colisões para investigar.'
assert len(cv) == 5
assert np.allclose(cv['f1'], pd.to_numeric(leg.iloc[:5]['f1']), rtol=0, atol=1e-12)
assert int(lim.loc[lim['limiar'].eq(.50), 'enderecos_sinalizados'].iloc[0]) == 9

checks = pd.DataFrame([
    ('Paridade das features / n_tx', '7 de 7 combinações de referência', 'CONFIRMADO NOS RELATÓRIOS'),
    ('Março/2025', '1 combinação sem feature de referência', 'NÃO COMPARÁVEL'),
    ('Colisões entre pools', '0 registradas na auditoria 12.02B', 'SEM COLISÕES OBSERVADAS'),
    ('Validação cruzada', '5 folds iguais ao legado (F1)', 'PARIDADE CONFIRMADA'),
    ('2020: limiar 0,50', '9 alertas; 3 pools conhecidos', 'TOTAL AGREGADO REPRODUZIDO'),
], columns=['Verificação', 'Achado', 'Leitura'])
display(checks)

## 3 · Gráficos para entender o que aconteceu

**Gráfico A — para onde foram as transações?** Em 2020 os pools conhecidos aparecem como destinatários diretos no dump. No recorte auditado de 2026, as operações foram atribuídas aos pools usando o **hash das chamadas**, enquanto a transação de topo apontava para um Router.

**Gráfico B — as features realmente bateram?** Mostra a quantidade de combinações com paridade completa, e aquela sem referência (2025). **Não** mostra percentual de cobertura da blockchain.

**Gráfico C — refatorar mudou a validação cruzada?** Compara o F1 fold a fold entre as duas versões; coincidência comprova equivalência das métricas internas, mas não generalização para novos contratos.

In [ ]:
#@title 3. Visualizar as evidências (três gráficos)
graficos = {}
fig, ax = plt.subplots(figsize=(9, 4))
g = atr[atr['periodo'].isin(['2020-10-29','2026-01-15'])].copy()
g['amostra'] = g['periodo'].str[:4] + ' · ' + g['pool']
g = g.sort_values(['periodo','pool'])
x = np.arange(len(g))
ax.bar(x, g['direta'], label='Destinatário direto')
ax.bar(x, g['via_chamada'], bottom=g['direta'], label='Atribuída pela chamada / Router')
ax.set_xticks(x, g['amostra'], rotation=35, ha='right')
ax.set_ylabel('Transações associadas ao pool')
ax.set_title('A · Atribuição de transações nos dias auditados')
ax.legend()
fig.tight_layout()
graficos[f'grafico_{len(graficos)+1}.png'] = fig
plt.show()

fig, ax = plt.subplots(figsize=(7, 3))
contagens = par.groupby(['periodo','status']).size().unstack(fill_value=0)
contagens.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('B · Paridade registrada por período')
ax.set_xlabel('Dia da auditoria')
ax.set_ylabel('Pools/dia')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Estado', fontsize=8)
fig.tight_layout()
graficos[f'grafico_{len(graficos)+1}.png'] = fig
plt.show()

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(cv['fold'], cv['f1'], marker='o', label='Notebook consolidado 01')
ax.plot(cv['fold'], pd.to_numeric(leg.iloc[:5]['f1']), marker='x', linestyle='--', label='Notebook legado')
ax.set(xticks=[1,2,3,4,5], ylim=(0,1.05), xlabel='Fold', ylabel='F1', title='C · Validação cruzada: paridade do F1')
ax.legend()
fig.tight_layout()
graficos[f'grafico_{len(graficos)+1}.png'] = fig
plt.show()

## 4 · Interpretação: o que podemos concluir?

**O que as evidências sustentam:**
- As **4 features** e a quantidade de transações coincidiram nas **7 combinações período/pool com referência processada** auditadas anteriormente.
- A diferença de atribuição **direta versus Router** explica por que contar apenas destinatários do topo em 2026 perde atividade dos pools.
- A nova organização do código preservou as **5 métricas F1 por fold** do legado; no limiar 0,50, manteve **9 alertas agregados em 2020, sendo 3 pools conhecidos**.

**O que elas NÃO sustentam:**
- Não comprovam ausência de Router em 2020, pois faltam chamadas equivalentes coletadas.
- Não demonstram que o modelo funciona em qualquer contrato Ethereum ou em todos os anos 2020–2026.
- Não permitem chamar alertas adicionais de “falsos positivos” sem rótulos verificados.
- Não substituem um teste **independente a partir dos dumps** ou comparação dos **endereços exatos** das previsões.

**Pergunta de estudo:** se a coleta de 2020 usa destinatário direto e a de 2026 usa chamadas internas, estamos vendo uma mudança no mixer ou no método de coleta? **Ainda não podemos separar os efeitos com rigor.**

## 5 · Registrar as evidências (opcional)

Esta célula salva um **manifesto novo, uma tabela de checks e os gráficos** em uma subpasta exclusiva do **seu** `Projeto_Mixers_Ethereum/results`. Não toca nos relatórios legados.

Para apenas estudar, deixe `SALVAR_NO_DRIVE = False`. Quando quiser guardar uma execução documentada, altere para `True`.

In [ ]:
#@title 4. Salvar auditoria e gráficos no Drive próprio (opcional)
SALVAR_NO_DRIVE = False  #@param {type:"boolean"}
if SALVAR_NO_DRIVE:
    from googleapiclient.http import MediaFileUpload
    pasta_raiz_id = '1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI'  # results do seu Drive
    pasta_local = Path('/content/auditoria04')
    pasta_local.mkdir(exist_ok=True)
    checks.to_csv(pasta_local / 'checks.csv', index=False)
    for nome_figura, figura in graficos.items():
        figura.savefig(pasta_local / nome_figura, dpi=160, bbox_inches='tight')
    manifest = {'experimento':'04_auditoria_visual',
                'data_utc':datetime.now(timezone.utc).isoformat(),
                'origem':'relatorios 12.02B + reproducao consolidada 01',
                'fontes_drive':ARQUIVOS,
                'sha256_fontes':{k:hashlib.sha256(v).hexdigest() for k,v in brutos.items()},
                'resultados':{'paridade_com_referencia':7,'sem_referencia':1,'colisoes_registradas':0},
                'limitacao':'auditoria dos relatórios; sem recomputar transações brutas'}
    (pasta_local/'manifesto.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
    # Criar uma pasta nova e única, evitando sobrescrever artefatos históricos.
    nome = '04_auditoria_visual_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    pasta_id = api.files().create(body={'name':nome,'mimeType':'application/vnd.google-apps.folder',
                              'parents':[pasta_raiz_id]},fields='id').execute()['id']
    for arq in pasta_local.iterdir():
        tipo = {'json':'application/json','csv':'text/csv','png':'image/png'}[arq.suffix.lstrip('.')]
        api.files().create(body={'name':arq.name,'parents':[pasta_id]},
            media_body=MediaFileUpload(str(arq),mimetype=tipo),fields='id').execute()
    print('Relatórios de auditoria:', 'https://drive.google.com/drive/folders/'+pasta_id)
else:
    print('Execução de estudo: nenhum arquivo gravado. Para salvar, marque SALVAR_NO_DRIVE.')

### Próximo marco científico

Com a auditoria documental consolidada, o passo seguinte é **reconstruir um conjunto histórico com protocolo de coleta constante** (múltiplos dias e múltiplos pools), validar rótulos e separar períodos de treino/teste. Não reutilizar 2020 e 2026 previamente inspecionados como se fossem testes cegos. A comparação dos **endereços exatos sinalizados** também continua pendente.